# k-Means Clustering

### Sebastien Huelke

Here we will explore a class of unsupervised machine learning models: clustering algorithms.
Clustering algorithms seek to learn, from the properties of the data, an optimal division or discrete labeling of groups of points.

Many clustering algorithms are available in Scikit-Learn and elsewhere, but perhaps the simplest to understand is an algorithm known as *k-means clustering*, which is implemented in ``sklearn.cluster.KMeans``.

We begin with some imports:

In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()  # for plot styling
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import itertools
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn import preprocessing
from sklearn.decomposition import PCA

## Excercise 1: 
Import the Candy data and show the first 10 rows.  Write a dataset description based on the information provided.  

In [ ]:
# Candy data from Rosie (/data/cs2300/L5)
candy_path = '/data/cs2300/L5/candy-data.csv'
df = pd.read_csv(candy_path)
print(df.shape)
df.head(10)

## Dataset Description
This is FiveThirtyEight's **Ultimate Halloween Candy Power Ranking** dataset (Walt Hickey, 2017). FiveThirtyEight built a website that showed visitors two randomly chosen fun-sized candies and asked which one they would prefer as a trick-or-treat haul. About 269,000 of these head-to-head matchups were collected. The dataset has **85 rows (one per candy) and 13 columns**:

| Column | Type | Meaning |
|---|---|---|
| `competitorname` | text | Name of the candy |
| `chocolate` | 0/1 | Does it contain chocolate? |
| `fruity` | 0/1 | Is it fruit flavored? |
| `caramel` | 0/1 | Is there caramel in the candy? |
| `peanutyalmondy` | 0/1 | Does it contain peanuts, peanut butter or almonds? |
| `nougat` | 0/1 | Does it contain nougat? |
| `crispedricewafer` | 0/1 | Does it contain crisped rice, wafers, or a cookie component? |
| `hard` | 0/1 | Is it a hard candy? |
| `bar` | 0/1 | Is it a candy bar? |
| `pluribus` | 0/1 | Is it one of many candies in a bag or box? |
| `sugarpercent` | 0–1 | Sugar content, as a percentile within this dataset |
| `pricepercent` | 0–1 | Unit price, as a percentile compared to the rest of the set |
| `winpercent` | 0–100 | Percentage of its matchups that the candy won |

Two of the "candies" are actually coins ("One dime" and "One quarter"), included as controls; they have 0 for every candy attribute. Note that `winpercent` is on a 0–100 scale while every other numeric column is on a 0–1 scale. This matters for k-means, which uses Euclidean distance (see Exercise 4).

## Exercise 2:

Create a scatter plot comparing the price percent and win percent.  Write a few sentences in the reflection below that describes what you observe (are there any obvious groups?).  

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df['pricepercent'], df['winpercent'], s=50)
plt.xlabel('pricepercent (price percentile)')
plt.ylabel('winpercent (% of matchups won)')
plt.title('Candy price percentile vs. win percent')
plt.show()

print('Correlation between pricepercent and winpercent:',
      round(df['pricepercent'].corr(df['winpercent']), 3))

### Reflection
There are no obviously separate groups in this plot; the points form one diffuse cloud. There is a weak positive trend (correlation ≈ 0.35): cheap candies (price percentile below ~0.35) almost all win less than 60% of their matchups, while most of the highest win percentages (above ~65%) belong to candies in the 0.5–0.9 price range. Price is not a strong predictor on its own, though: some of the most expensive candies have among the lowest win percentages (bottom right). The points also stack in vertical columns because `pricepercent` takes only a limited set of percentile values, which can make the plot look more grouped than it really is.

## Exercise 3: 
In the following cell, convert the dataframe into a 2D numpy array called X that only contains numerical data (by dropping the one categorical column "competitorname" - this column does not contain relevant info for clustering). 

In [ ]:
X = df.drop(columns=['competitorname']).to_numpy()
print(X.shape)
print(df.drop(columns=['competitorname']).columns.tolist())

The following code will create a kMeans classifier using the data in ``X``.  Run this cell, and feel free to come back to experiment with different values of k.  You should not need to modify this cell if you completed the previous cell as expected.  

In [ ]:
kmeans = KMeans(n_clusters=2)
kmeans.fit(X)
y_kmeans = kmeans.predict(X)

Let's visualize the results by plotting the data colored by these labels using the columns `pricepercent` and `winpercent`.
We will also plot the cluster centers in yellow as determined by the *k*-means estimator:

In [ ]:
plt.scatter(X[:, 10], X[:, 11], c=y_kmeans, s=50)

centers = kmeans.cluster_centers_
plt.scatter(centers[:, 10], centers[:, 11], c='blue', s=200, alpha=0.5);

## Exercise 4: 
This clustering isn't necessarily what you would expect, right?  This is most likely due to scaling bias in the distance calculation.  To address this, we need to scale the axes.  In the following cell, scale values using sklearn.preprocessing.scale(). You should re-do the clustering and plot the results to compare them to what you generated earlier. 

**Note: All further clustering should be done on the scaled data**

In [ ]:
X_scaled = preprocessing.scale(X)   # each column -> mean 0, std 1

kmeans_scaled = KMeans(n_clusters=2, n_init=10, random_state=42)
y_kmeans_scaled = kmeans_scaled.fit_predict(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Unscaled result from above, for comparison
axes[0].scatter(X[:, 10], X[:, 11], c=y_kmeans, s=50)
axes[0].scatter(kmeans.cluster_centers_[:, 10], kmeans.cluster_centers_[:, 11], c='blue', s=200, alpha=0.5)
axes[0].set(title='k=2, unscaled data', xlabel='pricepercent', ylabel='winpercent')

# Scaled result
axes[1].scatter(X_scaled[:, 10], X_scaled[:, 11], c=y_kmeans_scaled, s=50)
axes[1].scatter(kmeans_scaled.cluster_centers_[:, 10], kmeans_scaled.cluster_centers_[:, 11],
                c='blue', s=200, alpha=0.5)
axes[1].set(title='k=2, scaled data', xlabel='pricepercent (scaled)', ylabel='winpercent (scaled)')
plt.show()

# What distinguishes the two scaled clusters? (cluster centers in original units)
cols = df.columns.drop('competitorname')
summary = df[cols].groupby(y_kmeans_scaled).mean().round(2)
summary.insert(0, 'n_candies', np.bincount(y_kmeans_scaled))
summary

### Reflection:

How do these results compare with your unscaled results?

Unscaled, k-means with k=2 split the candies with an almost horizontal line at roughly 52–55% `winpercent` (centers near 40% and 64%), and `pricepercent` played almost no role. This is the scaling bias: `winpercent` spans about 62 units (22–84), while every other feature is in [0, 1], so `winpercent` dominates the Euclidean distance and the clustering is effectively one-dimensional.

After scaling (every column mean 0, std 1), all 12 features contribute equally, and the clusters no longer form clean bands in the price/win plot; they overlap, because the split is now driven mostly by the binary ingredient features. The table above shows the two scaled clusters are essentially **chocolate vs. non-chocolate**: one cluster (31 candies) is 97% chocolate and 68% bars, with mean price percentile 0.70 and mean win 63.8%; the other (54 candies) is 70% fruity, has no bars, is 69% pluribus, and has mean price percentile 0.34 and mean win 42.6%. This is a more meaningful grouping than "high win vs. low win." In the price/win plane the chocolate cluster still sits up and to the right, since chocolate candies tend to cost more and win more.

## Exercise 5: 
Try (and display graphs of) three additional k values to see how KMeans clusters the data.  Additionally, apply Principal Components Analysis (PCA) to give a "good" view of your 11-dimensional data in 2 dimensions (see the unsupervised learning notebook from lecture for an exmample on how to do this).  Document your experiments and write a short description in the reflection section below - which set of k-means clusters appeared to give the most distinct separation between groups?

In [ ]:
# Project the 11 scaled features down to 2 dimensions for viewing
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print('Variance explained by PC1, PC2:', pca.explained_variance_ratio_.round(3),
      ' total =', round(pca.explained_variance_ratio_.sum(), 3))

k_values = [2, 3, 4, 5]          # k=2 from Exercise 4 plus three additional values
models = {}
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, k in zip(axes.flat, k_values):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels_k = km.fit_predict(X_scaled)
    models[k] = (km, labels_k)
    centers_pca = pca.transform(km.cluster_centers_)
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_k, cmap='tab10', vmin=0, vmax=9, s=50)
    ax.scatter(centers_pca[:, 0], centers_pca[:, 1], c='black', marker='X', s=200, alpha=0.7)
    ax.set(title=f'k = {k}  (cluster sizes: {np.bincount(labels_k).tolist()})',
           xlabel='PC1', ylabel='PC2')
plt.tight_layout()
plt.show()

# Which features drive the two principal components?
pd.DataFrame(pca.components_.T, index=cols, columns=['PC1', 'PC2']).round(2)

In [ ]:
# Candies in each cluster for k = 2..5 (helps interpret the plots above)
for k in k_values:
    labels_k = models[k][1]
    print(f'=== k = {k} ===')
    for c in range(k):
        names = df.loc[labels_k == c, 'competitorname'].tolist()
        print(f'  cluster {c} ({len(names)}): ' + ', '.join(names[:8]) + (' ...' if len(names) > 8 else ''))

### Reflection
I tried k = 2, 3, 4 and 5 and plotted each on the first two principal components. These two components explain only about 47% of the variance (36% + 11%), so the 2-D view is a partial picture. PC1 loads positively on chocolate, bar, pricepercent and winpercent and negatively on fruity, pluribus and hard, so it is roughly a "chocolate bar vs. fruity candy" axis.

- **k = 2** gives the most distinct separation. The two clusters sit on either side of a visible gap along PC1, and they correspond to chocolate vs. non-chocolate candies.
- **k = 3** keeps the non-chocolate group (51) together and splits the chocolate candies into a 7-candy nougat/caramel bar cluster (3 Musketeers, Baby Ruth, Milky Way, Snickers, Payday, …) in the upper right and the remaining 27 chocolates.
- **k = 4** also splits off a 7-candy crisped-rice/wafer cluster (Kit Kat, Twix, 100 Grand, Krackel, Nestle Crunch, …). In the PCA view these points overlap the other chocolate clusters.
- **k = 5** splits the non-chocolate candies into a hard-candy cluster (Dum Dums, Gobstopper, Jawbusters, Lemonhead, Nerds, …) and the rest, but in 2-D the boundary runs through a continuous cloud with no gap.

The extra clusters for k ≥ 3 are interpretable (they match real ingredient combinations), but only k = 2 gives groups that are clearly separated visually.

## Excercise 6:

One way discussed in class to evaluate how "good" one group of clusters is is to compare the Davies-Bouldin score. Compute and display the Davies-Bouldin score for the 4 values of 'k' that you used above. Be sure to clearly label which output values belong to which graphs. 

Does the 'k' that the Davies Bouldin score denotes as creating the best clusters align with your "eye-balled" analysis from Exercise 5? If not, why might this be?

In [ ]:
db_scores = {}
for k in k_values:
    labels_k = models[k][1]
    db_scores[k] = davies_bouldin_score(X_scaled, labels_k)
    print(f'k = {k}: Davies-Bouldin score = {db_scores[k]:.3f}')

best_db_k = min(db_scores, key=db_scores.get)
print(f'\nLowest (best) Davies-Bouldin score: k = {best_db_k}')

### Reflection

Davies-Bouldin scores (lower is better): k=2: 1.475, k=3: 1.397, **k=4: 1.318**, k=5: 1.356. By this metric k = 4 gives the best clusters.

This does **not** match my visual judgement from Exercise 5, where k = 2 looked the most separated. Some likely reasons:
1. **The plots show only 2 of 12 dimensions.** The PCA view keeps ~47% of the variance. The small k=4 clusters (nougat bars, crisped-rice/wafer bars) are separated by features like `nougat` and `crispedricewafer` that barely show up in PC1/PC2, so they look overlapping in the plot even though they are distinct in the full space.
2. **Binary features make tight small clusters.** The 7 candies in each of those small clusters share nearly identical 0/1 feature values, so their within-cluster scatter is very small. Davies-Bouldin averages a per-cluster ratio, so a couple of very compact small clusters lower the average even though the large 50-candy cluster is spread out.
3. **The differences are small.** All scores fall between 1.32 and 1.48, and all are above 1. None of these clusterings is strongly separated, so the "best" k by this metric is not a strong signal and can change with the random initialization.

## Exercise 7

Another way to evaluate which number of clusters is best is to use the "within cluster sum of squares" or inertia.  It is a measure of how coherent the clusters are.  We can look at a variety of k-values to see how the inertia changes for different k-values.  This is not a perfect measure (it is subject to several types of biases), but it can be used to compare different k-values.  

To use the "Elbow Method" you want to find an "elbow" where there is a distinct change in the plot of different k-values.  It is not always very obvious, but this describes what to look for: https://www.geeksforgeeks.org/elbow-method-for-optimal-value-of-k-in-kmeans/ 

Calculate the "inertia" for k-means clusters with k varying between 2 and 6 and display and "elbow plot". Using the "elbow" method, which number of clusters would you say is best? Why? (put answer in reflection below).

Note that for each clustering object you create, it has an attribute called "inertia_" that represents this value. 

In [ ]:
k_range = range(2, 7)
inertias = []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)
    inertias.append(km.inertia_)
    print(f'k = {k}: inertia = {km.inertia_:.1f}')

print('\nDrop in inertia from k-1 to k:', {k: round(float(d), 1) for k, d in zip(list(k_range)[1:], -np.diff(inertias))})

plt.figure(figsize=(7, 5))
plt.plot(list(k_range), inertias, 'o-')
plt.xlabel('k (number of clusters)')
plt.ylabel('Inertia (within-cluster sum of squares)')
plt.title('Elbow plot, scaled candy data')
plt.xticks(list(k_range))
plt.show()

## Reflection
## Reflection
Inertia for k = 2…6: 712.4, 619.5, 536.6, 457.2, 410.5. The decrease at each step is 92.8, 82.9, 79.4, then 46.7.

There is no sharp elbow. The curve is close to a straight line from k = 2 to k = 5, so each added cluster removes about the same amount of within-cluster variance. The only noticeable bend is at **k = 5**, after which the improvement drops by roughly half (79 → 47). By the elbow method I would pick k = 5, but it is a weak elbow. Inertia always decreases as k grows, and a nearly linear curve suggests the data does not have a strong natural cluster structure. With binary features, the clusters mostly correspond to combinations of ingredients rather than well-separated blobs.

## Exercise 8

A third metric used to help identify how many clusters you should have is called the Silhouette Coefficient. You can read up on it in the following locations: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html and https://towardsdatascience.com/silhouette-coefficient-validating-clustering-techniques-e976bb81d10c.

Calculate and display the Silhouette Coefficient for k-means clusters with k varying between 2 and 6. Which k-value does the Silhouette Coefficient claim creates the best clusters?

**Answer here:** Silhouette coefficients: k=2: 0.303, k=3: 0.297, k=4: 0.295, k=5: 0.262, k=6: 0.263. The silhouette coefficient says **k = 2** gives the best clusters, which agrees with the visual judgement from Exercise 5. However, k = 3 and 4 are nearly tied, and all values are around 0.3 (on a scale where values near 1 mean well-separated clusters), which indicates weak, overlapping structure. The three metrics disagree (silhouette → 2, Davies-Bouldin → 4, elbow → about 5). That disagreement is itself evidence that this dataset has no single "natural" number of clusters.

In [ ]:
sil_scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels_k = km.fit_predict(X_scaled)
    sil_scores[k] = silhouette_score(X_scaled, labels_k)
    print(f'k = {k}: silhouette coefficient = {sil_scores[k]:.3f}')

best_sil_k = max(sil_scores, key=sil_scores.get)
print(f'\nHighest (best) silhouette coefficient: k = {best_sil_k}')

plt.figure(figsize=(7, 4))
plt.bar(list(sil_scores.keys()), list(sil_scores.values()))
plt.xlabel('k'); plt.ylabel('Silhouette coefficient')
plt.title('Silhouette coefficient vs. k')
plt.show()

# Example: k-means on images

To start, let's take a look at applying *k*-means on simple handwritten digits images.  Here we will attempt to use *k*-means to try to identify similar digits *without using the original label information*.  We will start by loading the digits and then finding the ``KMeans`` clusters.

The digits data consist of 1,797 samples with 64 features, where each of the 64 features is the brightness of one pixel in an 8×8 image.  We start by loading the data

In [1]:
from sklearn.datasets import load_digits
digits = load_digits()

Let's take a look at one of the images.  

In [ ]:
plt.imshow(digits.images[0], cmap='gray')
plt.show()

## Exercise 9:
Write a few lines of code to create a KMeans model using ``digits.data`` and 10 clusters.  No need to plot, just create the clusters for now. Note that you should name your model object `kmeans` and the assigned clusters to `clusters` so that the following code cells will run appropriately.

In [ ]:
kmeans = KMeans(n_clusters=10, n_init=10, random_state=0)
clusters = kmeans.fit_predict(digits.data)
kmeans.cluster_centers_.shape

Let's see what the cluster centers look like:

In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(8, 3))
centers = kmeans.cluster_centers_.reshape(10, 8, 8)
for axi, center in zip(ax.flat, centers):
    axi.set(xticks=[], yticks=[])
    axi.imshow(center, interpolation='nearest', cmap=plt.cm.binary)

We see that *even without the labels*, ``KMeans`` is able to cluster whose centers are recognizable digits, with the exception of 1 and 8.  

*k*-means knows nothing about the identity of the cluster, so the clusters are not in any specific order.  To identify the actual order, we can use the labels in the following code.  

In [ ]:
from scipy.stats import mode

labels = np.zeros_like(clusters)
for i in range(10):
    mask = (clusters == i) #array of booleans for each cluster group
    labels[mask] = mode(digits.target[mask])[0] #get "mode" of each cluster group's label

Now we can check how accurate our unsupervised clustering was in finding similar digits within the data:

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(digits.target, labels)

With just a simple *k*-means algorithm, we discovered the correct grouping for 80% of the input digits!
Let's check the confusion matrix for this:

In [ ]:
from sklearn.metrics import confusion_matrix
mat = confusion_matrix(digits.target, labels)
sns.heatmap(mat.T, square=True, annot=True, fmt='d', cbar=False,
            xticklabels=digits.target_names,
            yticklabels=digits.target_names)
plt.xlabel('true label')
plt.ylabel('predicted label')

As we might expect from the cluster centers we visualized before, the main point of confusion is between the eights and ones.
But this still shows that using *k*-means, we can essentially build a digit classifier *without reference to any known labels*!

## Exercise 10:
Let's try this with higher resolution images!  Use the code below to import the 28x28 DIGITS images. Note, you will either need to be on Rosie's Tensorflow 2.x container, or you will likely need to install some libraries locally.

Then complete the following steps:
1. Plot an example image 
2. Train a KMeans cluster using the testing data.  Note that you will need to reshape the data to be in the correct format. Note that the N images will result in an N x 28 x 28 array. You will need to reshape the data to be an N x (28\*28) array. Look into the numpy .reshape() method to accomplish this.
3. Plot the cluster centers
4. Calculate the accuracy
5. Plot the confusion matrix
6. Use the training data and re-run the experiment to see how sensitive the model is to the data.  
7. Write a short reflection on what you observed from this and why it might be the case.  

In [ ]:
import tensorflow as tf
from tensorflow import keras
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
from scipy.stats import mode

print('Train:', X_train.shape, ' Test:', X_test.shape)

# 1. Plot an example image
plt.imshow(X_test[0], cmap='gray')
plt.title(f'Example test image, label = {y_test[0]}')
plt.axis('off')
plt.show()

def run_mnist_kmeans(images, targets, name):
    """Cluster N x 28 x 28 images into 10 groups and evaluate against the true labels."""
    # 2. Reshape N x 28 x 28 -> N x 784 and train k-means
    data = images.reshape(images.shape[0], 28 * 28).astype(np.float32) / 255.0
    km = KMeans(n_clusters=10, n_init=10, random_state=0)
    cl = km.fit_predict(data)

    # 3. Plot the cluster centers
    fig, ax = plt.subplots(2, 5, figsize=(8, 3))
    for axi, center in zip(ax.flat, km.cluster_centers_.reshape(10, 28, 28)):
        axi.set(xticks=[], yticks=[])
        axi.imshow(center, interpolation='nearest', cmap=plt.cm.binary)
    fig.suptitle(f'Cluster centers ({name} data)')
    plt.show()

    # Map each cluster to the most common true label in it
    lab = np.zeros_like(cl)
    mapping = []
    for i in range(10):
        mask = (cl == i)
        digit = np.atleast_1d(mode(targets[mask])[0])[0]
        lab[mask] = digit
        mapping.append(int(digit))
    print(f'{name}: cluster -> digit mapping:', mapping)
    print(f'{name}: digits with no cluster of their own:', sorted(set(range(10)) - set(mapping)))

    # 4. Accuracy
    acc = accuracy_score(targets, lab)
    print(f'{name}: accuracy = {acc:.3f}')

    # 5. Confusion matrix
    mat = confusion_matrix(targets, lab, labels=range(10))
    plt.figure(figsize=(7, 7))
    sns.heatmap(mat.T, square=True, annot=True, fmt='d', cbar=False,
                xticklabels=range(10), yticklabels=range(10))
    plt.xlabel('true label'); plt.ylabel('predicted label')
    plt.title(f'Confusion matrix ({name} data)')
    plt.show()
    return acc

acc_test = run_mnist_kmeans(X_test, y_test, 'test (10,000 images)')

In [ ]:
# 6. Re-run the same experiment on the training data
acc_train = run_mnist_kmeans(X_train, y_train, 'train (60,000 images)')
print(f'\nAccuracy on test data:  {acc_test:.3f}')
print(f'Accuracy on train data: {acc_train:.3f}')

### Reflection
**Test data (10,000 images):** accuracy ≈ 0.59. **Training data (60,000 images):** accuracy ≈ 0.59. That is much lower than the ~0.79 on the 8×8 scikit-learn digits.

**Sensitivity to the data.** Overall accuracy barely changed between the 10,000-image test set and the 6× larger training set, but the clusters that formed did change. On the test set, two clusters became "1" (upright and slanted) and two became "7". On the training set, two clusters became "0" (round and slanted) and two became "1". Both times, **no cluster was assigned to 5 or 9**, so every 5 and 9 was automatically misclassified. The confusion matrices show where the errors come from: 4/7/9 are merged (they share a vertical stroke, and the "9"-looking centers end up labeled 7 or 4), 3/5/8 are mixed, and many digits fall into the "1" cluster.

**Why.** k-means groups images by Euclidean distance between raw pixel vectors, so it finds the largest sources of pixel variation, such as stroke slant, thickness and position, and these are not the same as digit identity. A slanted 1 and an upright 1 can be farther apart in pixel space than a 4 and a 9. With only 10 clusters, spending two on variants of one digit forces other digits to share a cluster. More data does not fix this, because the limitation is the representation (raw pixels + Euclidean distance + spherical clusters), not the sample size. The 8×8 digits did better because the low resolution blurs away much of the variation in individual handwriting, and that dataset is smaller and more uniform. Note also that this "accuracy" is optimistic for an unsupervised method, since it uses the true labels to name each cluster.